<a href="https://colab.research.google.com/github/Vishali-Vivek006/Engineering-employability-prediction/blob/main/notebooks/05_resume_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!pip install -q pymupdf python-docx sentence-transformers pandas scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 14.4 MB/s eta 0:00:00


In [2]:

from google.colab import files
import fitz
from docx import Document
import os
import re

uploaded = files.upload()
resume_filename = next(iter(uploaded))

def extract_resume_text(filepath):
    ext = os.path.splitext(filepath)[1].lower()

    if ext == ".pdf":
        with fitz.open(filepath) as pdf:
            text = "\n".join(page.get_text() for page in pdf)

    elif ext == ".docx":
        doc = Document(filepath)
        parts = [p.text for p in doc.paragraphs]

        for table in doc.tables:
            for row in table.rows:
                parts.append(
                    " | ".join(cell.text for cell in row.cells)
                )

        text = "\n".join(parts)

    else:
        raise ValueError("Upload a PDF or DOCX resume.")

    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

resume_text = extract_resume_text(resume_filename)

print("Extracted characters:", len(resume_text))
print("\n--- RESUME TEXT ---\n")
print(resume_text[:8000])

if not resume_text:
    print("No text found. The PDF may need OCR.")


Saving Vishali_Resume__1___1_.pdf to Vishali_Resume__1___1_.pdf
Extracted characters: 1860

--- RESUME TEXT ---

VISHALI V
+91 6379742355
•
vishalivanith2006@gmail.com
LinkedIn: https://www.linkedin.com/in/vishali-vivekanandan-4876a42a4
GITHUB: https://github.com/Vishali-Vivek006
EDUCATION
• B.Tech Information Technology, MIT Campus Anna University, Chennai, Tamilnadu
Expected 2027
CGPA 8.55 (As of 6 semesters)
• Shepherds Matric.Hr.Sec.School,Tirupattur,Tamilnadu
Graduated 2023
Class 12: Secured 97.6%
SKILLS
Technical Skills
Algorithm and Data Structures, Database management, OOPS, Operating System.
Computer Networks,Machine Learning,Java Spring Boot
Programming Languages
C, C++,Python,Java.
Web development
Front-end: HTML, CSS, JS,React. Back-end: Node.js. Database: SQL.
PROJECTS
Optimized Word Lookup System
• Developed Command- Line dictionary that uses trie for efficient word queries.Enabled prefix-based word search and
autocorrection for misspelled words.
• Technologies involved: 

In [3]:

profile = {
    "education_mentions": [],
    "cgpa_mentions": [],
    "project_section": "",
    "internship_section": "",
    "certification_section": ""
}

# Education keywords
education_patterns = [
    r"\bB\.?\s?Tech\b",
    r"\bB\.?\s?E\.?\b",
    r"\bM\.?\s?Tech\b",
    r"\bMCA\b",
    r"\bM\.?\s?Sc\b",
    r"\bMBA\b",
    r"\bBachelor(?:'s)?\b",
    r"\bMaster(?:'s)?\b"
]

for pattern in education_patterns:
    matches = re.findall(pattern, resume_text, flags=re.IGNORECASE)
    profile["education_mentions"].extend(matches)

# CGPA/GPA explicitly mentioned in the resume
profile["cgpa_mentions"] = re.findall(
    r"\b(?:CGPA|GPA)\s*[:\-]?\s*\d+(?:\.\d+)?",
    resume_text,
    flags=re.IGNORECASE
)

# Extract text between section headings
def get_section(text, start_words, end_words):
    lines = text.splitlines()
    capture = False
    collected = []

    for line in lines:
        clean = line.strip()
        if not clean:
            continue

        normalized = re.sub(r"[^a-z ]", "", clean.lower()).strip()

        if any(word in normalized for word in start_words):
            capture = True
            continue

        if capture and any(word in normalized for word in end_words):
            break

        if capture:
            collected.append(clean)

    return "\n".join(collected)

profile["project_section"] = get_section(
    resume_text,
    ["projects", "academic projects"],
    ["education", "skills", "internship", "certifications", "achievements"]
)

profile["internship_section"] = get_section(
    resume_text,
    ["internship", "experience", "work experience"],
    ["education", "skills", "projects", "certifications", "achievements"]
)

profile["certification_section"] = get_section(
    resume_text,
    ["certifications", "certificates"],
    ["education", "skills", "projects", "internship", "achievements"]
)

for field, value in profile.items():
    print(f"\n--- {field.upper()} ---")
    print(value if value else "Not detected")



--- EDUCATION_MENTIONS ---
['B.Tech']

--- CGPA_MENTIONS ---
['CGPA 8.55']

--- PROJECT_SECTION ---
Optimized Word Lookup System
• Developed Command- Line dictionary that uses trie for efficient word queries.Enabled prefix-based word search and
autocorrection for misspelled words.
• Technologies involved: C, Trie, Levenshtein distance Algorithm,Edit distance Algorithm.
Web platform for Dance school
• Developed a full-stack website for a dance academy, using the MERN stack, featuring course listings, instructor
profiles, event registration, testimonial carousel, and secure user form handling with MongoDB integration.
• Technologies involved: React, Node.js, MongoDB,Express.
Traffic Congestion Prediction
• Developed a machine learning-based traffic congestion prediction system using Random Forest, Decision Tree, and
Logistic Regression to classify congestion levels from traffic and weather data.
• Technologies involved: Python,Machine Learning.
CHESS GAME
• CHESS GAME Developed a CHESS 

In [4]:

skill_dictionary = {
    "Programming": [
        "Python", "Java", "C", "C++", "C#", "JavaScript",
        "TypeScript", "R", "Go", "Kotlin", "PHP", "Ruby",
        "Swift", "Dart"
    ],
    "Web Development": [
        "HTML", "CSS", "React", "Angular", "Vue",
        "Node.js", "Express.js", "REST API", "Bootstrap"
    ],
    "Database": [
        "SQL", "MySQL", "PostgreSQL", "MongoDB",
        "SQLite", "Oracle", "Redis"
    ],
    "CS Fundamentals": [
        "Data Structures", "Algorithms", "DSA", "OOP",
        "Operating Systems", "Computer Networks", "DBMS"
    ],
    "AI ML Data": [
        "Machine Learning", "Deep Learning", "NumPy",
        "Pandas", "Matplotlib", "Scikit-learn",
        "TensorFlow", "PyTorch", "NLP"
    ],
    "Cloud DevOps Tools": [
        "AWS", "Azure", "Google Cloud", "Docker",
        "Kubernetes", "Git", "GitHub", "Linux", "Jenkins"
    ]
}

def extract_skills(text, dictionary):
    found = {}
    for category, skills in dictionary.items():
        found[category] = []

        for skill in skills:
            pattern = r"(?<!\w)" + re.escape(skill) + r"(?!\w)"
            if re.search(pattern, text, flags=re.IGNORECASE):
                found[category].append(skill)

    return found

extracted_skills = extract_skills(resume_text, skill_dictionary)

for category, skills in extracted_skills.items():
    print(f"{category}: {skills}")

all_skills = sorted({
    skill
    for skills in extracted_skills.values()
    for skill in skills
})

print("\nTotal unique skills:", len(all_skills))


Programming: ['Python', 'Java', 'C', 'C++']
Web Development: ['HTML', 'CSS', 'React', 'Node.js']
Database: ['SQL', 'MySQL', 'MongoDB']
CS Fundamentals: ['Data Structures', 'Computer Networks']
AI ML Data: ['Machine Learning', 'Deep Learning']
Cloud DevOps Tools: ['GitHub']

Total unique skills: 16


In [5]:

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# Example role requirements; replace with your verified role catalogue.
role_requirements = [
    "Develop frontend applications using React and JavaScript",
    "Build backend services and integrate REST APIs",
    "Manage relational databases using SQL",
    "Use Git for version control and collaboration"
]

# Use resume project and experience text as evidence.
resume_evidence = [
    section for section in [
        profile["project_section"],
        profile["internship_section"],
        resume_text
    ] if section
]

# For a first baseline, compare each requirement to the full resume.
resume_embedding = embedding_model.encode(
    [resume_text],
    normalize_embeddings=True
)

requirement_embeddings = embedding_model.encode(
    role_requirements,
    normalize_embeddings=True
)

similarities = cosine_similarity(
    requirement_embeddings,
    resume_embedding
).flatten()

for requirement, score in zip(role_requirements, similarities):
    print(f"{score:.3f} | {requirement}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

0.122 | Develop frontend applications using React and JavaScript
0.091 | Build backend services and integrate REST APIs
0.154 | Manage relational databases using SQL
0.006 | Use Git for version control and collaboration


In [6]:

import pandas as pd
import json

output = {
    "resume_filename": resume_filename,
    "education_mentions": profile["education_mentions"],
    "cgpa_mentions": profile["cgpa_mentions"],
    "technical_skills": all_skills,
    "skills_by_category": extracted_skills,
    "project_section": profile["project_section"],
    "internship_section": profile["internship_section"],
    "certification_section": profile["certification_section"]
}

with open("resume_extraction_output.json", "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

pd.DataFrame({
    "Skill": all_skills
}).to_csv("extracted_skills.csv", index=False)

print("Saved resume_extraction_output.json")
print("Saved extracted_skills.csv")


Saved resume_extraction_output.json
Saved extracted_skills.csv


In [7]:

files.download("resume_extraction_output.json")
files.download("extracted_skills.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:

import re

ENGINEERING_SKILLS = {
    "CSE_IT": {
        "Programming": [
            "Python", "Java", "C++", "C#", "JavaScript",
            "TypeScript", "C", "R", "Go", "PHP", "Kotlin"
        ],
        "Web_Development": [
            "HTML", "CSS", "React", "Angular", "Vue.js",
            "Node.js", "Express.js", "REST API", "Bootstrap"
        ],
        "Database": [
            "SQL", "MySQL", "PostgreSQL", "MongoDB",
            "Oracle", "SQLite", "Redis"
        ],
        "CS_Fundamentals": [
            "Data Structures", "Algorithms", "DSA", "OOP",
            "Operating Systems", "Computer Networks", "DBMS"
        ],
        "AI_ML_Data": [
            "Machine Learning", "Deep Learning", "NumPy",
            "Pandas", "Matplotlib", "Scikit-learn",
            "TensorFlow", "PyTorch", "NLP"
        ],
        "Cloud_DevOps": [
            "AWS", "Azure", "Google Cloud", "Docker",
            "Kubernetes", "Git", "GitHub", "Linux", "Jenkins"
        ]
    },

    "ECE": {
        "Electronics": [
            "Digital Electronics", "Analog Electronics",
            "Electronic Circuits", "Circuit Design",
            "VLSI", "CMOS", "Verilog", "VHDL",
            "FPGA", "PCB Design"
        ],
        "Embedded_Systems": [
            "Embedded Systems", "Arduino", "Raspberry Pi",
            "Microcontrollers", "Microprocessors",
            "ARM", "8051", "Embedded C", "IoT"
        ],
        "Signals_Communication": [
            "Digital Signal Processing", "Signal Processing",
            "Communication Systems", "Wireless Communication",
            "Antenna Design", "RF Engineering",
            "MATLAB", "Simulink"
        ],
        "Testing_Design": [
            "Oscilloscope", "Circuit Simulation",
            "LTspice", "Proteus", "Cadence"
        ]
    },

    "EEE": {
        "Electrical_Systems": [
            "Power Systems", "Electrical Machines",
            "Electrical Circuit Analysis", "Power Generation",
            "Power Transmission", "Power Distribution",
            "Electrical Protection", "Switchgear"
        ],
        "Control_Power_Electronics": [
            "Power Electronics", "Control Systems",
            "PLC", "SCADA", "Industrial Automation",
            "Motor Control", "Drives",
            "MATLAB", "Simulink"
        ],
        "Electrical_Design": [
            "ETAP", "AutoCAD Electrical",
            "Electrical Wiring", "Renewable Energy",
            "Solar PV", "Energy Management"
        ]
    },

    "Mechanical": {
        "CAD_CAM": [
            "AutoCAD", "SolidWorks", "CATIA",
            "Creo", "Fusion 360", "NX CAD",
            "CAM", "CNC", "GD&T"
        ],
        "Simulation_Analysis": [
            "ANSYS", "Finite Element Analysis", "FEA",
            "Computational Fluid Dynamics", "CFD",
            "Solid Mechanics", "Thermal Analysis"
        ],
        "Core_Mechanical": [
            "Thermodynamics", "Fluid Mechanics",
            "Heat Transfer", "Machine Design",
            "Manufacturing Processes", "Robotics",
            "Materials Science", "Quality Control",
            "Six Sigma", "Lean Manufacturing"
        ]
    },

    "Civil": {
        "Civil_Design": [
            "AutoCAD", "Revit", "STAAD.Pro",
            "ETABS", "Civil 3D", "BIM",
            "Structural Analysis", "Structural Design",
            "RCC Design", "Steel Structure Design"
        ],
        "Construction": [
            "Quantity Surveying", "Estimation and Costing",
            "Project Planning", "Construction Management",
            "Primavera P6", "MS Project",
            "Site Supervision", "Quality Control"
        ],
        "Core_Civil": [
            "Surveying", "Total Station",
            "Geotechnical Engineering", "Soil Mechanics",
            "Transportation Engineering", "Hydraulics",
            "Environmental Engineering"
        ]
    },

    "Aerospace": {
        "Aircraft_Design": [
            "Aerodynamics", "Flight Mechanics",
            "Aircraft Structures", "Aircraft Design",
            "Propulsion", "Gas Turbines",
            "Orbital Mechanics", "Flight Dynamics"
        ],
        "Simulation_Analysis": [
            "ANSYS", "CATIA", "MATLAB", "Simulink",
            "Computational Fluid Dynamics", "CFD",
            "Finite Element Analysis", "FEA",
            "Abaqus", "OpenFOAM"
        ]
    }
}


def extract_domain_skills(resume_text, domain):
    """Extract skills from the selected engineering domain."""
    if domain not in ENGINEERING_SKILLS:
        raise ValueError(
            f"Unsupported domain: {domain}. "
            f"Choose from {list(ENGINEERING_SKILLS)}"
        )

    results = {}

    for category, skill_list in ENGINEERING_SKILLS[domain].items():
        matched = []

        for skill in skill_list:
            # Avoid matching C inside words such as CATIA.
            pattern = r"(?<!\w)" + re.escape(skill) + r"(?!\w)"

            if re.search(pattern, resume_text, flags=re.IGNORECASE):
                matched.append(skill)

        results[category] = matched

    return results


def flatten_skills(skills_by_category):
    return sorted({
        skill
        for skills in skills_by_category.values()
        for skill in skills
    })


# Example: use the resume_text variable from Phase 5.
# Change the domain to test another engineering department.

selected_domain = "Civil"

skills_by_category = extract_domain_skills(
    resume_text,
    selected_domain
)

extracted_skills = flatten_skills(skills_by_category)

print("Selected domain:", selected_domain)
print("Extracted skills:", extracted_skills)

for category, skills in skills_by_category.items():
    print(f"\n{category}: {skills}")


Selected domain: Civil
Extracted skills: []

Civil_Design: []

Construction: []

Core_Civil: []
